# Process Stage — Clean, Verify, Document, and Report

**General explanation:** This Process phase turns dirty data into clean, trustworthy, and usable data. It emphasises checking for errors, cleaning and transforming the data, verifying the results, documenting every change, reporting the outcome, and improving the work after feedback.  
  
**Project workflow role:** This notebook will apply the prepared data from public DataCo Smart Supply Chain proxy files generated by `01_prepare_data_readiness.ipynb` and will create the validated inputs required by the analytical questions frozen in `00_define_problems.ipynb`.  
  
## Delivery Analytics Capstone  
  
**General explanation:** The following table presents the metadata of this notebook that makes the artefact independently understandable during an offline interview review.  
  
**Project workflow role:** This opening identifies the intended notebook, audience, phase, and claim boundary before any cleaning logic is later inserted.  
   
**Portfolio boundary:** This independent case study uses public proxy data. It does not claim access to, endorsement by, or evidence about any organisation's actual systems, customers, operations, policies, or performance.  

## Notebook purpose and course-workflow coverage

This section defines the single Process-stage question, the planned evidence, and the boundary that prevents cleaning from drifting into exploratory analysis or modelling.  
  
This notebook will answer one Process-stage question:  
  
**Can the supplied DataCo source be cleaned, transformed, verified, documented, and physically exported as privacy-safe order-level and complete-week inputs for AQ1–AQ4?**  
  
The implemented notebook should eventually produce:  
  
- a clean order-level table with one eligible order per row;  
- a clean complete-week table containing only base weekly inputs;  
- evidence that raw and processed counts, keys, target values, and additive totals reconcile;  
- documentation of cleaning decisions, transformations, unresolved issues, and limitations;  
- an actual on-disk handoff that Analyze can read and independently accept.  

## Confirmed reference inputs for the framework

**General Explanation:** Process should remain connected to the original business task and inspect the actual data and metadata. Reference facts should guide the cleaning plan without being mistaken for completed cleaning evidence.  
  
**Project workflow role:** This section records the constraints that the supplied `00` notebook, raw main CSV, and field-description CSV impose on the later implementation.  
  
| Reference | Confirmed design input | Consequence for this framework |
|---|---|---|
| `00_define_problems.ipynb` | One unique order is the decision grain | Item rows should be reconciled and aggregated to one order |
| `00_define_problems.ipynb` | Binary late-delivery outcome; prediction time is order creation | Target construction and post-order leakage must be separated |
| `00_define_problems.ipynb` | Complete Monday-to-Sunday weeks; four-week horizon | Process prepares complete-week inputs but does not forecast |
| `00_define_problems.ipynb` | Public-proxy, human-decision-support boundary | No organisational, causal, implementation, savings, or ROI claim |
| Main CSV | 180,519 order-item rows, 53 observed fields, and 65,752 distinct `Order Id` values | Cleaning must distinguish legitimate item repetition from duplicate rows |
| Main CSV | One to five item rows occur per order; `Order Item Id` identifies the raw row grain | `Order Id` must not be used as a row-deduplication key |
| Main CSV | Missingness is concentrated in four columns, including a fully missing product description and a highly incomplete order postcode | Missing-value treatment must be field-specific; blanket imputation is inappropriate |
| Field-description CSV | 52 documented fields; `Order Zipcode` is observed but undocumented | The schema gap must be recorded and the unsupported field must not be used without justification |
| Both raw files | Direct identifiers, a credential-like field, precise location fields, order outcomes, and post-order timing fields are present | Privacy removal and prediction-time field-use boundaries are necessary before analytical export |

The later code must reproduce and display these facts from the working files; this Markdown framework is not a substitute for that evidence.


## 1. Establish the Process task and protect the source  
  
**General Explanation:** Before cleaning, I should understand the objective, choose an appropriate tool, keep the business task in view, and preserve the original data so changes can be checked and reversed.  
  
**Project workflow role:** This section    
- converts the frozen **Ask** rules and **Prepare** findings into a concise processing contract  
- establishes a safe source-to-output path.  
  
**Planned workflow:** Receive stage requirements → confirm the allowed Process scope → justify the tool → protect raw files → create a working copy and named output locations.


### 1.1 Receive the Process handoff

The Prepare phase established the analytical scope, source files, and
data-use constraints required for processing.

The Process phase will:

- inspect and clean the supplied DataCo order-item dataset;
- verify its schema, metadata coverage, and row grain;
- remove sensitive and unnecessary fields;
- create privacy-safe order-item, order-level, and weekly analytical
  tables; and
- validate the resulting tables before they are used in the Analyze phase.

Data grain alteration:  
- Raw data grain：one **order item** per row  
- Processed data grain: one **order** per row 


Because DataCo is a proxy dataset, subsequent findings will be presented
as evidence from the supplied data rather than as direct claims.

### 1.2 Define the cleaning and validation plan

Processing will follow a reproducible sequence from source inspection to
validated analytical outputs:

1. **Inspect the source data**  
   Confirm the file structure, metadata coverage, candidate keys, row grain,
   and fields required for the planned analysis.

2. **Assess data quality**  
   Review missing values, duplicate records, invalid categories, date fields,
   delivery measures, quantities, and monetary values.

3. **Clean and standardise the data**  
   Remove sensitive and unnecessary fields, standardise column names and data
   types, and apply documented treatments to identified quality issues.

4. **Build analytical tables**  
   Create privacy-safe order-item, order-level, and weekly tables at clearly
   defined grains. Order-level delivery measures will be calculated only after
   order-item records have been aggregated and reconciled.

5. **Validate and export the outputs**  
   Check schema, privacy, row grain, aggregation totals, weekly calendar
   completeness, and exported-file readability before confirming readiness
   for the Analyze phase.

Each validation will be performed where the relevant output is created.
Readiness for analysis will therefore be determined only after all processing
and final validation steps are complete.

### 1.3 Select tools for data processing

Python with pandas was selected to clean, standardise, transform, and validate
the supplied order-item data. A code-based workflow is appropriate for the
dataset's size and makes each processing decision reproducible and traceable.

Pandas will be used to:

- inspect data-quality issues;
- remove sensitive and unnecessary fields;
- standardise values and data types;
- create order-item, order-level, and weekly analytical tables; and
- validate the processed outputs before analysis.

The original CSV files will remain unchanged. All transformations will be
applied to a separate working copy.

### 1.4 Load the source data

The supplied order-item dataset and data dictionary are loaded from the
raw-data directory. The main dataset is assigned to `working_df`, which will
be used throughout the Process phase.

Processing results will be exported separately to the processed-data
directory. Detailed checks of the schema, metadata coverage, row grain, and
data quality begin in Section 2.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


# Identify the project root.
# The notebook may run from either the project root or the notebooks folder.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Define the project folders and source files.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MAIN_PATH = RAW_DIR / "DataCoSupplyChainDataset.csv"
DICTIONARY_PATH = RAW_DIR / "DescriptionDataCoSupplyChain.csv"

In [2]:
# Confirm that the required source files are available.
required_files = [MAIN_PATH, DICTIONARY_PATH]
missing_files = [path.name for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Required source file(s) not found in data/raw: "
        + ", ".join(missing_files)
    )

In [3]:
# Create the folder for processed outputs.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# Load the source data.
working_df = pd.read_csv(
    MAIN_PATH,
    encoding="latin-1",
    low_memory=False,
)

dictionary_df = pd.read_csv(
    DICTIONARY_PATH,
    encoding="latin-1",
)

In [4]:
# Preview the loaded data.
display(working_df.head())
display(dictionary_df.head())

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


,FIELDS,DESCRIPTION
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased...
3,Benefit per order,: Earnings per order placed
4,Sales per customer,: Total sales per customer made per customer


## 2. Inspect the source data

Before cleaning begins, the source data is inspected to understand  
- its structure  
- metadata coverage  
- expected row grain 
- potential data-quality issues  
  
This section    
- examines       
    - the structure of the dataset    
    - the raw data grain  
- assesses 
    - the quality issues that may require treatment before analytical tables are created.

### 2.1 Inspect source structure, metadata coverage, and row grain

The main dataset and its accompanying data dictionary are inspected before any
columns or records are changed.

This inspection will:

- confirm the number of rows and columns;
- review column names and data types;
- compare dataset fields with the supplied data dictionary;
- test whether `Order Item Id` uniquely identifies each row; and
- confirm whether multiple item rows can belong to the same order.

The supplied dataset is expected to contain one order item per row. This row
grain will be treated as confirmed only if `Order Item Id` is complete and unique,
while `Order Id` can appear across multiple item rows.

**Number of rows and columns**

In [5]:
# Display the size of each loaded table.
print("Main dataset shape:", working_df.shape)
print("Data dictionary shape:", dictionary_df.shape)

Main dataset shape: (180519, 53)
Data dictionary shape: (52, 2)


**Column names and data types**

In [6]:
# Review the column names and data types in the main dataset.
structure_summary = pd.DataFrame({
    "Column": working_df.columns,
    "Data type": working_df.dtypes.astype(str).values # The one-to-one relationship between each column name and its data type already exists in working_df.dtypes.
})

display(structure_summary)

,Column,Data type
0,Type,str
1,Days for shipping (real),int64
2,Days for shipment (scheduled),int64
3,Benefit per order,float64
4,Sales per customer,float64
5,Delivery Status,str
6,Late_delivery_risk,int64
7,Category Id,int64
8,Category Name,str
9,Customer City,str


**Field comparison of dataset and description**

In [7]:
# Get the documented field names from the first column
# of the supplied data dictionary.
documented_fields = (
    dictionary_df.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.strip()
)

dataset_fields = pd.Index(working_df.columns.str.strip())


# Identify fields that do not match between the two files.
fields_without_documentation = sorted(
    set(dataset_fields) - set(documented_fields)
)

documented_fields_not_in_data = sorted(
    set(documented_fields) - set(dataset_fields)
)


# Summarise metadata coverage.
metadata_summary = pd.DataFrame({
    "Check": [
        "Fields in the main dataset",
        "Fields listed in the data dictionary",
        "Dataset fields without matching documentation",
        "Documented fields not found in the dataset"
    ],
    "Count": [
        len(dataset_fields),
        len(documented_fields),
        len(fields_without_documentation),
        len(documented_fields_not_in_data)
    ]
})

display(metadata_summary)

# Display the specific field-name differences.
field_difference_details = pd.DataFrame({
    "Mismatch type": (
        ["Dataset field without matching documentation"]
        * len(fields_without_documentation)
        +
        ["Documented field not found in the dataset"]
        * len(documented_fields_not_in_data)
    ),
    "Field": (
        fields_without_documentation
        + documented_fields_not_in_data
    )
})

display(field_difference_details)

,Check,Count
0,Fields in the main dataset,53
1,Fields listed in the data dictionary,52
2,Dataset fields without matching documentation,2
3,Documented fields not found in the dataset,1


,Mismatch type,Field
0,Dataset field without matching documentation,Order Zipcode
1,Dataset field without matching documentation,shipping date (DateOrders)
2,Documented field not found in the dataset,Shipping date (DateOrders)


**Metadata mismatch assessment and follow-up actions**

The field comparison identified three unmatched entries. However, these entries represent
two underlying metadata issues rather than three separate fields.

| Field or field pair | Mismatch type | Importance | Assessment |
|---|---|---|---|
| `Order Zipcode` | Dataset field without matching documentation | Medium | The field exists in the dataset but has no corresponding definition in the supplied data dictionary. Its meaning and expected format should be confirmed before it is used in analysis. |
| `shipping date (DateOrders)` / `Shipping date (DateOrders)` | Case-related naming mismatch | Low | These names appear to refer to the same field. The mismatch occurs because the dataset uses a lowercase `s`, while the data dictionary uses an uppercase `S`. Python's exact string comparison is case-sensitive. |

The `Order Zipcode` mismatch is more important because the absence of documentation creates
uncertainty about the field's definition, expected values, and analytical use. In particular,
a ZIP code should normally be treated as a categorical location identifier rather than a
numerical measure, even if it is stored as a numeric data type.

The shipping-date mismatch does not indicate that the field or its documentation is missing.
It is a metadata naming inconsistency and does not require changes to the field values.

The following actions will be taken:

1. Record `shipping date (DateOrders)` and `Shipping date (DateOrders)` as the same field,
   with a difference in letter case only.
2. Use the actual dataset column name, `shipping date (DateOrders)`, in the processing code
   unless column names are standardised later.
3. Review the values and data type of `Order Zipcode` to understand its structure.
4. Confirm the meaning of `Order Zipcode` using the source documentation or the business
   context before including it in analysis.
5. Treat `Order Zipcode` as a categorical field and preserve possible leading zeros if it is
   retained.
6. Document the confirmed definition of `Order Zipcode` in the project's data dictionary or
   data-cleaning notes.
7. Do not remove either field solely because of these metadata mismatches.

Therefore, the comparison reveals one genuine documentation gap and one minor naming
inconsistency. Neither issue currently provides evidence that the corresponding dataset
columns should be deleted.

**Uniqueness of `Order Item Id` in raw data**

In [8]:
# Test whether Order Item Id uniquely identifies each source row.
row_count = len(working_df)

unique_order_item_ids = working_df["Order Item Id"].nunique(
    dropna=True
)

missing_order_item_ids = working_df["Order Item Id"].isna().sum()

duplicated_order_item_rows = working_df["Order Item Id"].duplicated(
    keep=False
).sum()


order_item_id_summary = pd.DataFrame({
    "Check": [
        "Total rows",
        "Unique Order Item Id values",
        "Missing Order Item Id values",
        "Rows with duplicated Order Item Id values"
    ],
    "Result": [
        row_count,
        unique_order_item_ids,
        missing_order_item_ids,
        duplicated_order_item_rows
    ]
})

display(order_item_id_summary)

,Check,Result
0,Total rows,180519
1,Unique Order Item Id values,180519
2,Missing Order Item Id values,0
3,Rows with duplicated Order Item Id values,0


In [9]:
# Confirm whether Order Item Id uniquely identifies each row.
order_item_id_is_unique = (
    missing_order_item_ids == 0
    and duplicated_order_item_rows == 0
    and unique_order_item_ids == row_count
)

if order_item_id_is_unique:
    print(
        "Confirmed: Order Item Id is complete and uniquely "
        "identifies each source row."
    )
else:
    print(
        "Not confirmed: Order Item Id contains missing or "
        "duplicated values and requires further review."
    )

Confirmed: Order Item Id is complete and uniquely identifies each source row.


### 2.2 Assess data quality issues  
  
After confirming the source structure and row grain, the dataset is assessed for data-quality issues that may affect the analysis.  
  
This assessment will:  
  
- identify missing values and measure their frequency;
- check for fully duplicated rows;
- review descriptive statistics for numerical fields;
- inspect important categorical fields used in the business questions;
- check whether key date fields can be interpreted as dates; and
- Check selected business rules to identify potentially invalid or unusual values for further investigation.  
  
The purpose of this section is to identify and document potential issues before making cleaning decisions. 

#### 2.2.1 Missing Values

**Assessment**

In [10]:
# Calculate missing-value counts and percentages for every column.
missing_summary = pd.DataFrame({
    "Column": working_df.columns,
    "Missing values": working_df.isna().sum().values,
    "Missing percentage": (
        working_df.isna().mean().mul(100).values
    )
})

# Keep only columns containing missing values.
missing_summary = (
    missing_summary[
        missing_summary["Missing values"] > 0
    ]
    .sort_values(
        by="Missing percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

missing_summary["Missing percentage"] = (
    missing_summary["Missing percentage"].round(4)
)

display(missing_summary)

,Column,Missing values,Missing percentage
0,Product Description,180519,100.0000
1,Order Zipcode,155679,86.2397
2,Customer Lname,8,0.0044
3,Customer Zipcode,3,0.0017


**Findings**

Four fields contain missing values, but their analytical importance and likely causes differ.

- `Product Description` contains 180,519 missing values, representing 100% of the source records. The field provides no usable information in its current form.
- `Order Zipcode` contains 155,679 missing values, representing 86.2397% of the records. This is substantial missingness, but it may be structural because postal-code availability and applicability can vary by country or market. Its geographic missingness pattern requires further investigation.
- `Customer Lname` contains 8 missing values, representing 0.0044% of the records. Although the missingness is negligible, the field contains personally identifiable information and is not required for the planned analysis.
- `Customer Zipcode` contains 3 missing values, representing 0.0017% of the records. The missingness is negligible, but ZIP codes should not be statistically imputed because doing so could create false geographic information.
- The very low missing percentages in `Customer Lname` and `Customer Zipcode` would appear as 0.00% if rounded to two decimal places; therefore, both counts and sufficiently precise percentages are reported.

These missing values do not affect the verification of the source row grain because
`Order Item Id` and `Order Id` are not missing. However, the relevance, privacy
implications, and missingness pattern of each affected field must be considered before
cleaning decisions are implemented.

**Proposed treatment**

1. Remove `Product Description` from the analytical dataset because it is entirely
   missing and contributes no information to the business questions.
2. Examine `Order Zipcode` missingness by `Market`, `Order Region`, and
   `Order Country` to determine whether it represents structural non-applicability
   rather than an accidental data-quality problem.
3. Do not impute missing `Order Zipcode` values. If the field is not required by the
   business questions and its high missingness limits its usefulness, remove it from
   the analytical tables after the geographic assessment is completed.
4. Do not impute `Customer Lname`. Remove the entire field when creating the
   privacy-safe analytical dataset because customer surnames are personally
   identifiable and are not analytically necessary.
5. Review the three records with missing `Customer Zipcode`. Do not replace the
   missing values with a mean, median, mode, or inferred ZIP code, as this could
   introduce inaccurate location information.
6. Remove or exclude `Customer Zipcode` from privacy-safe analytical tables if
   customer-level postal information is not required. If retained for a justified
   geographic analysis, preserve the missing values and treat the field as a
   categorical location code rather than a numerical measure.
7. Record all column removals and retained missing values in the data-cleaning log.

No rows will be removed solely because of the missing values identified above. The
treatments will be applied at the column or field level where appropriate, preserving
valid order-item records for subsequent order-level reconciliation.

#### 2.2.2 Duplicate rows

**Assessment**

In [11]:
# Check for fully duplicated source rows.
fully_duplicated_rows = working_df.duplicated(
    keep=False
).sum()

extra_duplicate_copies = working_df.duplicated().sum()


duplicate_summary = pd.DataFrame({
    "Check": [
        "Rows belonging to fully duplicated groups",
        "Additional duplicate copies"
    ],
    "Result": [
        fully_duplicated_rows,
        extra_duplicate_copies
    ]
})

display(duplicate_summary)

,Check,Result
0,Rows belonging to fully duplicated groups,0
1,Additional duplicate copies,0


**Findings**

The duplicate-row assessment found no fully duplicated source records.

- `Rows belonging to fully duplicated groups` is **0**, meaning no rows share identical values across all columns.
- `Additional duplicate copies` is also **0**, meaning there are no redundant copies that would need to be removed.
- This result supports the integrity of the source data at its expected order-item grain.
- This check is distinct from the `Order Item Id` uniqueness assessment. The current check compares complete rows across all fields, whereas the earlier uniqueness check determines whether `Order Item Id` uniquely identifies each source row.

**Proposed treatment**

1. Retain all source rows because no fully duplicated records were identified.
2. Do not apply `drop_duplicates()` to the dataset, as it would not remove any records and is unnecessary.
3. Preserve repeated `Order Id` values because they may legitimately represent multiple order items belonging to the same order and are not evidence of duplicated rows.
4. Continue using `Order Item Id` to identify individual source records and `Order Id` to reconcile them into the order-level analytical table.
5. Record the duplicate-row check as passed in the data-quality assessment.

No duplicate-related cleaning action is required.

#### 2.2.3 Numerical fields

**Assessment**

In [12]:
# Review descriptive statistics for numerical columns.
numeric_summary = (
    working_df
    .select_dtypes(include="number")
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "Column"})
)

display(numeric_summary)

,Column,count,mean,std,min,25%,50%,75%,max
0,Days for shipping (real),180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
1,Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
2,Benefit per order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
3,Sales per customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
4,Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
5,Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
6,Customer Id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
7,Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
8,Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
9,Latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933


**Findings**

The descriptive-statistics review identified several important patterns in the
numerical fields.

- Most numerical fields contain 180,519 non-missing values. `Customer Zipcode`
  contains 180,516 non-missing values, while `Order Zipcode` contains only 24,840,
  which is consistent with the missing-value assessment.
- `Product Description` has no non-missing values. It appears in the numerical
  summary because an entirely missing column may be inferred as a numerical data
  type, but its numerical statistics have no analytical meaning.
- Several numerical columns are identifiers, geographic codes, or category codes,
  including `Order Id`, `Order Item Id`, `Customer Id`, `Category Id`,
  `Department Id`, `Product Card Id`, and the ZIP-code fields. Their means,
  standard deviations, and quartiles are not interpreted as business measures.
- `Days for shipping (real)` ranges from 0 to 6 days, with a mean of approximately
  3.50 days. `Days for shipment (scheduled)` ranges from 0 to 4 days, with a mean
  of approximately 2.93 days. These ranges do not show negative shipping durations,
  but zero-day values require contextual validation.
- `Order Item Quantity` ranges from 1 to 5, so no zero or negative quantities are
  indicated by the summary.
- `Order Item Discount Rate` ranges from 0 to 0.25, which is consistent with a
  proportional discount rate between 0% and 25%.
- `Order Item Discount` ranges from 0 to 500. The maximum is substantially above
  the upper quartile of 29.99 and should be examined together with product price,
  quantity, sales, and discount rate before being classified as an error.
- `Order Item Product Price` ranges from 9.99 to 1,999.99, while `Sales` ranges
  from 9.99 to 1,999.99. These high values may represent expensive products rather
  than invalid observations and require product-level context.
- `Order Profit Per Order` and `Benefit per order` range from approximately
  -4,274.98 to 911.80. Negative values may represent genuine losses and should not
  automatically be treated as invalid. The wide range and large standard deviation
  indicate that the profit distribution requires further investigation.
- `Order Item Profit Ratio` ranges from -2.75 to 0.50. Negative ratios are possible
  for loss-making transactions, but the minimum value should be reviewed with the
  associated sales, discounts, and profit values.
- `Sales per customer` and `Order Item Total` have identical descriptive
  statistics. `Benefit per order` and `Order Profit Per Order` also have identical
  descriptive statistics. This suggests possible duplicate or equivalent fields,
  but row-level equality must be tested before any column is removed.
- `Order Item Cardprod Id` and `Product Card Id` also have identical descriptive
  statistics. Their relationship should be checked directly rather than inferred
  from summary statistics alone.
- `Product Status` is 0 for every record and therefore has no variation. Its field
  definition and analytical usefulness should be confirmed.
- Latitude ranges from approximately -33.94 to 48.78, and longitude ranges from
  approximately -158.03 to 115.26. These values fall within the general valid
  coordinate limits, although consistency with the associated cities and countries
  has not yet been verified.

The descriptive statistics identify investigation candidates rather than confirmed
data errors. Extreme values are not considered invalid solely because they are far
from the mean or quartiles.

**Proposed treatment**

1. Exclude identifier, category-code, and postal-code fields from distributional
   interpretation. Retain them only where they are required for joining, grouping,
   validation, or geographic analysis.
2. Remove `Product Description` during cleaning because it is entirely missing,
   rather than treating it as a numerical measure.
3. Preserve the original numerical values during the assessment stage. Do not
   remove observations solely on the basis of a high standard deviation, an extreme
   minimum or maximum, or distance from the quartiles.
4. Inspect records containing zero actual or scheduled shipping days and confirm
   whether they represent valid same-day processing or delivery.
5. Examine the largest `Order Item Discount` values together with `Order Item
   Product Price`, `Order Item Quantity`, `Sales`, and `Order Item Discount Rate`
   to confirm their internal consistency.
6. Investigate the most negative and positive profit observations using their
   associated sales, discount, product, order status, and delivery information.
   Retain genuine loss-making transactions.
7. Verify whether `Sales per customer` equals `Order Item Total` for every row and
   whether `Benefit per order` equals `Order Profit Per Order` for every row.
   Remove a redundant field only if complete row-level equivalence and identical
   business meaning are confirmed.
8. Verify the relationship between `Order Item Cardprod Id` and `Product Card Id`
   before deciding whether both fields are required.
9. Review the definition of `Product Status`. If the field is correctly recorded
   but remains constant at zero and provides no analytical information, remove it
   from the final analytical tables and document the reason.
10. Validate coordinate fields against their associated country, state, and city
    where geographic analysis requires them.
11. Apply explicit business-rule checks in Section 2.2.6 to confirm numerical
    validity, including non-negative sales, positive quantities, valid discount-rate
    bounds, and non-negative shipping durations.
12. Before calculating order-level measures, confirm how fields labelled
    `per order` behave across multiple item rows belonging to the same `Order Id`.
    Values must not be summed repeatedly if the same order-level amount is duplicated
    across its item records.

No numerical values will be capped, transformed, corrected, or removed until their
business meaning and record-level context have been investigated.

#### 2.2.4 Important categorical fields

**Assessment**

In [13]:
# Select categorical fields relevant to the analysis.
categorical_columns = [
    "Delivery Status",
    "Shipping Mode",
    "Order Status",
    "Market",
    "Order Region",
    "Customer Segment",
    "Department Name",
    "Category Name"
]

# Summarise missing values and category counts.
categorical_summary = pd.DataFrame({
    "Missing values":
        working_df[categorical_columns].isna().sum(),
    "Unique categories":
        working_df[categorical_columns].nunique()
})

display(categorical_summary)

,Missing values,Unique categories
Delivery Status,0,4
Shipping Mode,0,4
Order Status,0,9
Market,0,5
Order Region,0,23
Customer Segment,0,3
Department Name,0,11
Category Name,0,50


In [14]:
# Review the frequency distribution of key business categories.
frequency_columns = [
    "Delivery Status",
    "Shipping Mode",
    "Market",
    "Customer Segment"
    #"Department Name"
    #"Category Name"
]

for column in frequency_columns:
    print(f"\n{column}")
    display(
        working_df[column]
        .value_counts()
        .rename_axis(column)
        .reset_index(name="Record count")
    )


Delivery Status


,Delivery Status,Record count
0,Late delivery,98977
1,Advance shipping,41592
2,Shipping on time,32196
3,Shipping canceled,7754



Shipping Mode


,Shipping Mode,Record count
0,Standard Class,107752
1,Second Class,35216
2,First Class,27814
3,Same Day,9737



Market


,Market,Record count
0,LATAM,51594
1,Europe,50252
2,Pacific Asia,41260
3,USCA,25799
4,Africa,11614



Customer Segment


,Customer Segment,Record count
0,Consumer,93504
1,Corporate,54789
2,Home Office,32226


In [15]:
# Check categorical labels for obvious spacing issues.
spacing_issues = []

for column in categorical_columns:

    issue_mask = (
        working_df[column].ne(
            working_df[column].str.strip()
        )
        |
        working_df[column].str.contains(
            "  ",
            na=False
        )
    )

    if issue_mask.any():

        issue_counts = (
            working_df.loc[issue_mask, column]
            .value_counts()
            .rename_axis("Original value")
            .reset_index(name="Record count")
        )

        issue_counts.insert(0, "Field", column)
        spacing_issues.append(issue_counts)

if spacing_issues:
    spacing_issue_summary = pd.concat(
        spacing_issues,
        ignore_index=True
    )

    display(spacing_issue_summary)
else:
    print("No obvious spacing issues were identified.")

,Field,Original value,Record count
0,Order Region,West of USA,7993
1,Order Region,US Center,5887
2,Order Region,South of USA,4045
3,Department Name,Health and Beauty,362
4,Category Name,Cameras,592
5,Category Name,Books,405
6,Category Name,CDs,271
7,Category Name,Baby,207
8,Category Name,As Seen on TV!,68


**Findings**

- No missing values were identified in the selected categorical fields.
- The number of categories varies across the assessed fields, reflecting
  different levels of business and geographic detail.
- Leading or trailing spaces were identified in selected values within
  `Order Region`, `Department Name`, and `Category Name`.
- Repeated internal spaces were also found in labels including
  `South of  USA` and `As Seen on  TV!`.
- Category frequencies represent order-item records rather than distinct orders.

**Proposed treatment**

- Remove leading and trailing spaces from categorical text fields.
- Replace confirmed repeated internal spaces with a single space.
- Retain all valid categories because category frequency alone does not indicate
  a data-quality problem.
- Recheck the affected category values after cleaning.

#### 2.2.5 Key dates

**Assessment**

In [16]:
# Select the key date fields used in the analysis.
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]


# Convert the date fields temporarily for assessment.
parsed_dates = working_df[date_columns].apply(
    lambda column: pd.to_datetime(
        column,
        format="%m/%d/%Y %H:%M",
        errors="coerce"
    )
)


# Summarise the original data types, parsing results,
# and date coverage.
date_summary = pd.DataFrame({
    "Original data type":
        working_df[date_columns].dtypes.astype(str),
    "Missing or invalid values":
        parsed_dates.isna().sum(),
    "Earliest date":
        parsed_dates.min(),
    "Latest date":
        parsed_dates.max()
})

date_summary.index.name = "Field"

display(date_summary)


,Original data type,Missing or invalid values,Earliest date,Latest date
Field,,,,
order date (DateOrders),str,0,2015-01-01,2018-01-31 23:38:00
shipping date (DateOrders),str,0,2015-01-03,2018-02-06 22:14:00


In [17]:
# Check whether any shipping date occurred before
# the corresponding order date.
shipping_before_order = (
    parsed_dates["shipping date (DateOrders)"]
    < parsed_dates["order date (DateOrders)"]
)

date_check_summary = pd.DataFrame({
    "Check": [
        "Shipping date before order date"
    ],
    "Records identified": [
        int(shipping_before_order.sum())
    ]
})

display(date_check_summary)

,Check,Records identified
0,Shipping date before order date,0


**Findings**

- Both date fields were originally stored as text, and all values were successfully
  converted to datetime for assessment.
- Order dates range from 1 January 2015 to 31 January 2018.
- Shipping dates range from 3 January 2015 to 6 February 2018.
- No shipping dates occurred before their corresponding order dates.

**Proposed treatment**

- Convert the order and shipping date fields to datetime during data cleaning.
- Retain all records because no missing, invalid, or chronologically inconsistent
  dates were identified.
- No date imputation or record removal is required.

#### 2.2.6 Business rules

**Assessment**

In [18]:
# Separate cancelled shipments from completed delivery outcomes.
cancelled_shipments = (
    working_df["Delivery Status"]
    .eq("Shipping canceled")
)

eligible_shipments = ~cancelled_shipments


# Calculate the expected late-delivery flag.
calculated_late_risk = (
    working_df["Days for shipping (real)"]
    > working_df["Days for shipment (scheduled)"]
).astype(int)


# Compare the calculated and recorded flags for eligible shipments.
late_risk_mismatch = (
    eligible_shipments
    & working_df["Late_delivery_risk"].ne(
        calculated_late_risk
    )
)


# Summarise the business-rule assessment.
business_rule_summary = pd.DataFrame({
    "Check": [
        "Shipping-cancelled order-item records",
        "Distinct cancelled orders",
        "Late-risk mismatches among non-cancelled records"
    ],
    "Count": [
        int(cancelled_shipments.sum()),
        int(
            working_df.loc[
                cancelled_shipments,
                "Order Id"
            ].nunique()
        ),
        int(late_risk_mismatch.sum())
    ]
})

display(business_rule_summary)

,Check,Count
0,Shipping-cancelled order-item records,7754
1,Distinct cancelled orders,2855
2,Late-risk mismatches among non-cancelled records,0


**Findings**

- A total of 7,754 order-item records, representing 2,855 distinct orders, were
  marked as `Shipping canceled`.
- Cancelled shipments do not represent completed delivery outcomes and should
  not be classified as on-time deliveries.
- After cancelled shipments were separated, `Late_delivery_risk` agreed with
  the comparison between actual and scheduled shipping days for all remaining
  records.

**Proposed treatment**

- Exclude orders with `Delivery Status = Shipping canceled` from the
  delivery-timeliness analysis.
- Retain cancelled orders separately if cancellation analysis is required later.
- Use `Late_delivery_risk` as the recorded late-delivery outcome for the
  remaining eligible orders.
- Apply this treatment during cleaning without modifying the raw dataset.

### 2.3 Identify privacy, security, and publication risks  
  
**General Explanation:** Protecting personal and sensitive information is part of responsible data processing. Data availability does not automatically justify analytical or public use.  
  
**Project workflow role:** The dataset contains  
- personal identifiers  
- detailed location information  
- password-related field  
- record identifiers.  
  
These fields are reviewed to determine what should be excluded from the analysis or retained only for internal processing.

**Assessment**

In [19]:
# Group fields that may create privacy or publication risks.
privacy_field_groups = [
    (
        "Direct identifiers and password-related field",
        [
            "Customer Email",
            "Customer Fname",
            "Customer Lname",
            "Customer Street",
            "Customer Password"
        ],
        "Exclude from analysis and publication"
    ),
    (
        "Customer identifiers and detailed locations",
        [
            "Customer Id",
            "Order Customer Id",
            "Customer City",
            "Customer State",
            "Customer Zipcode",
            "Order City",
            "Order State",
            "Order Zipcode",
            "Latitude",
            "Longitude"
        ],
        "Exclude unless specifically required"
    ),
    (
        "Order identifiers",
        [
            "Order Id",
            "Order Item Id"
        ],
        "Retain internally only"
    )
]


In [20]:
# Check which listed fields are present without displaying their values.
privacy_assessment_rows = []

for field_group, fields, treatment in privacy_field_groups:

    fields_found = [
        field
        for field in fields
        if field in working_df.columns
    ]

    privacy_assessment_rows.append({
        "Field group": field_group,
        "Fields found": len(fields_found),
        "Fields identified": ", ".join(fields_found),
        "Treatment": treatment
    })


privacy_assessment = pd.DataFrame(
    privacy_assessment_rows
)

display(privacy_assessment)

,Field group,Fields found,Fields identified,Treatment
0,Direct identifiers and password-related field,5,"Customer Email, Customer Fname, Customer Lname...",Exclude from analysis and publication
1,Customer identifiers and detailed locations,10,"Customer Id, Order Customer Id, Customer City,...",Exclude unless specifically required
2,Order identifiers,2,"Order Id, Order Item Id",Retain internally only


**Findings**

- Five fields containing direct customer identifiers or password-related
  information were identified.
- Ten customer-key or detailed-location fields were identified. Combining these
  fields with order information could increase identification risk.
- `Order Id` and `Order Item Id` are operational identifiers. They are useful for
  checking the dataset grain and calculating distinct orders, but do not need to
  appear in published outputs.
- No sensitive field values were displayed during this assessment.
- This assessment applies only to the public DataCo proxy dataset.

**Proposed treatment**

- Exclude names, email, street address and the password-related field from the
  analysis-ready dataset and all published outputs.
- Exclude customer identifiers and detailed-location fields unless they are
  specifically required for a justified calculation.
- Use broader geographic fields such as `Market`, `Order Region`, and
  `Order Country` for aggregated geographic analysis.
- Retain `Order Id` and `Order Item Id` only in internal working data when required
  for validation and order-level calculations.
- Keep the raw source file outside the public repository and publish only
  aggregated, non-identifying results.

### 2.4 Create the cleaning plan

**Purpose**

The findings from Sections 2.2 and 2.3 are summarised before cleaning begins.
The plan records the required action and how each result will be checked.

| Finding or requirement | Field(s) | Planned action | Verification |
|---|---|---|---|
| Inconsistent spacing was identified in category labels | `Order Region`, `Department Name`, `Category Name` | Remove leading, trailing, and confirmed repeated spaces | Recheck the affected category values |
| `Product Description` is completely missing and is not required for the analysis | `Product Description` | Remove the field from the analysis-ready dataset | Confirm that the field is absent |
| Key date fields are stored as text but can be successfully parsed | Order and shipping date fields | Convert the fields to datetime | Confirm that no values fail conversion |
| Cancelled shipments are not completed delivery outcomes | `Delivery Status` | Retain the records but exclude them from delivery-timeliness calculations | Confirm that cancelled records are absent from the eligible delivery subset |
| Personal and detailed-location fields are not required for the analysis | Direct identifiers, contact details, password-related and detailed-location fields | Exclude these fields from the analysis-ready dataset | Confirm that the excluded fields are absent |
| The source data has an order-item grain | `Order Id`, `Order Item Id` | Preserve valid item rows and aggregate by `Order Id` only when order-level analysis is required | Confirm one row per order in the order-level output |

## 3. Clean – resolve the detected issues

- Only the treatments confirmed in Section 2 are applied to `working_df`, including:    
    - Correct confirmed categorical spacing  
    - Convert key dates to datetime    
    - Remove unusable and sensitive fields  
- After cleaning, the dataset remains stored in `working_df` as a pandas DataFrame, with **one row per order item** as data grain.    
- The raw source files remain unchanged.  
- The cleaned `working_df` is used as the input for the transformations in Section 4.

### 3.1 Correct confirmed categorical spacing

**Purpose**

The spacing issues confirmed in Section 2.2.4 are corrected in:  
- `Order Region`  
- `Department Name`  
- `Category Name`  
  
Only leading, trailing, and repeated whitespace is corrected. Category wording and
capitalisation remain unchanged.

**Applied treatment**

In [21]:
# Fields with confirmed spacing issues.
spacing_columns = [
    "Order Region",
    "Department Name",
    "Category Name"
]


# Remove leading, trailing, and repeated whitespace.
for column in spacing_columns:
    working_df[column] = (
        working_df[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# Confirm that no spacing issues remain.
remaining_spacing_issues = (
    working_df[spacing_columns]
    .apply(
        lambda column: column.str.contains(
            r"^\s|\s$|\s{2,}",
            regex=True,
            na=False
        ).sum()
    )
    .rename("Remaining spacing issues")
    .to_frame()
)

display(remaining_spacing_issues)

,Remaining spacing issues
Order Region,0
Department Name,0
Category Name,0


**Result**

- The confirmed spacing issues were corrected in all three fields.
- No leading, trailing, or repeated whitespace remained after cleaning.
- Category wording and capitalisation were unchanged.

### 3.2 Convert key dates to datetime

**Purpose**

The two date fields assessed in Section 2.2.5 are converted from text to datetime
so they can be used in time-based analysis.

**Applied treatment**

In [22]:
# Date fields confirmed as valid in Section 2.2.5.
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]


# Convert the date fields from text to datetime.
working_df[date_columns] = (
    working_df[date_columns]
    .apply(
        lambda column: pd.to_datetime(
            column,
            format="%m/%d/%Y %H:%M"
        )
    )
)


# Verify the conversion.
date_conversion_summary = pd.DataFrame({
    "Data type":
        working_df[date_columns].dtypes.astype(str),
    "Missing values after conversion":
        working_df[date_columns].isna().sum()
})

date_conversion_summary.index.name = "Field"

display(date_conversion_summary)

,Data type,Missing values after conversion
Field,,
order date (DateOrders),datetime64[us],0
shipping date (DateOrders),datetime64[us],0


**Result**

- Both date fields were successfully converted to datetime.
- No missing values were introduced during conversion.
- No records were removed or imputed.

### 3.3 Remove unusable and sensitive fields
  
**Purpose**
  
The fully missing `Product Description` field and the privacy-related fields identified in Sections 2.2 and 2.3 are removed from `working_df`.
  
`Order Id` and `Order Item Id` are retained for validation and later aggregation.

**Applied Treatment**

In [23]:
# Fields confirmed for removal in Section 2.
fields_to_remove = [
    "Product Description",
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Password",
    "Customer Id",
    "Order Customer Id",
    "Customer City",
    "Customer State",
    "Customer Zipcode",
    "Order City",
    "Order State",
    "Order Zipcode",
    "Latitude",
    "Longitude"
]


# Remove the fields from the working DataFrame.
working_df.drop(
    columns=fields_to_remove,
    inplace=True
)


# Confirm the removal and retention of internal identifiers.
internal_id_fields = [
    "Order Id",
    "Order Item Id"
]

field_removal_summary = pd.DataFrame({
    "Check": [
        "Fields removed",
        "Removed fields still present",
        "Internal identifier fields retained"
    ],
    "Count": [
        len(fields_to_remove),
        sum(
            field in working_df.columns
            for field in fields_to_remove
        ),
        sum(
            field in working_df.columns
            for field in internal_id_fields
        )
    ]
})

display(field_removal_summary)

,Check,Count
0,Fields removed,16
1,Removed fields still present,0
2,Internal identifier fields retained,2


**Result**

- Sixteen unusable or privacy-related fields were removed from `working_df`.
- None of the excluded fields remained after cleaning.
- `Order Id` and `Order Item Id` were retained for internal processing.
- No rows were removed, so the order-item grain was preserved.

## 4. Transform - Prepare the order-level data

**Purpose**  
  
- Transform the cleaned order-item-level `working_df` into the analysis-ready datasets required for the Analyze phase.  
- Exclude cancelled shipments from the eligible delivery scope while retaining them in `working_df`.  
- Aggregate eligible `order-item` records by `Order Id` to create order_df, with one row per eligible order.  
- Use `order_df` to create `weekly_df`, containing only complete Monday-to-Sunday weeks and the base weekly measures required for later analysis.  
- Pass both datasets to Section 5 for final verification and saving.

### 4.1 Define the eligible delivery scope

**Purpose**

Cancelled shipments do not represent completed delivery outcomes and are excluded
from delivery-timeliness analysis.

They remain in `working_df`. A Boolean mask identifies the eligible order-item rows
used to create `order_df` in Section 4.2.

**Applied transformation**

In [24]:
# Identify rows eligible for delivery-timeliness analysis.
eligible_delivery_mask = (
    working_df["Delivery Status"]
    .ne("Shipping canceled")
)


# Summarise the eligible and excluded scopes.
delivery_scope_summary = pd.DataFrame({
    "Scope": [
        "Eligible deliveries",
        "Cancelled shipments excluded"
    ],
    "Order-item rows": [
        int(eligible_delivery_mask.sum()),
        int((~eligible_delivery_mask).sum())
    ],
    "Unique orders": [
        working_df.loc[
            eligible_delivery_mask,
            "Order Id"
        ].nunique(),
        working_df.loc[
            ~eligible_delivery_mask,
            "Order Id"
        ].nunique()
    ]
})

display(delivery_scope_summary)


# Confirm that no cancelled shipments enter the eligible scope.
assert not working_df.loc[
    eligible_delivery_mask,
    "Delivery Status"
].eq("Shipping canceled").any()

,Scope,Order-item rows,Unique orders
0,Eligible deliveries,172765,62897
1,Cancelled shipments excluded,7754,2855


**Result**

- Cancelled shipments were excluded from the eligible delivery scope but retained
  in `working_df`.
- `eligible_delivery_mask` identifies the order-item rows used in Section 4.2.
- No separate DataFrame was created at this stage.

### 4.2 Create the order-level dataset

**Purpose**  
  
- Aggregate eligible order-item rows `Order Id`  
- Create `order_df` with **one row per eligible order** as data grain.  
  
Fields confirmed as consistent within each order are retained once. Item and product counts are calculated, while item quantity and order value are summed.

**Applied transformation**

In [25]:
# Order-level fields confirmed as consistent within each order.
order_level_fields = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Late_delivery_risk",
    "Shipping Mode",
    "Type",
    "Customer Segment",
    "Market",
    "Order Region",
    "Days for shipment (scheduled)",
    "Days for shipping (real)"
]


# Define the order-level aggregation rules.
order_aggregation = {
    field: (field, "first")
    for field in order_level_fields
}

order_aggregation.update({
    "Order Item Count": (
        "Order Item Id",
        "nunique"
    ),
    "Product Count": (
        "Product Card Id",
        "nunique"
    ),
    "Order Quantity": (
        "Order Item Quantity",
        "sum"
    ),
    "Order Value": (
        "Order Item Total",
        "sum"
    )
})


# Aggregate eligible order-item rows to one row per order.
order_df = (
    working_df.loc[eligible_delivery_mask]
    .groupby(
        "Order Id",
        as_index=False
    )
    .agg(**order_aggregation)
    .sort_values([
        "order date (DateOrders)",
        "Order Id"
    ])
    .reset_index(drop=True)
)


# Validate the order-level output.
order_validation_summary = pd.DataFrame({
    "Check": [
        "One row per eligible order",
        "Order Id is unique",
        "Cancelled orders excluded",
        "Late_delivery_risk is complete",
        "Order quantity reconciles",
        "Order value reconciles"
    ],
    "Passed": [
        len(order_df)
        == working_df.loc[
            eligible_delivery_mask,
            "Order Id"
        ].nunique(),

        order_df["Order Id"].is_unique,

        not order_df["Order Id"].isin(
            working_df.loc[
                ~eligible_delivery_mask,
                "Order Id"
            ]
        ).any(),

        order_df["Late_delivery_risk"].notna().all(),

        order_df["Order Quantity"].sum()
        == working_df.loc[
            eligible_delivery_mask,
            "Order Item Quantity"
        ].sum(),

        round(order_df["Order Value"].sum(), 2)
        == round(
            working_df.loc[
                eligible_delivery_mask,
                "Order Item Total"
            ].sum(),
            2
        )
    ]
})

display(order_validation_summary)

assert order_validation_summary["Passed"].all()

,Check,Passed
0,One row per eligible order,True
1,Order Id is unique,True
2,Cancelled orders excluded,True
3,Late_delivery_risk is complete,True
4,Order quantity reconciles,True
5,Order value reconciles,True


In [26]:
# Preview the new order-level DataFrame.
display(order_df.head(3))

# Inspect its dimensions, fields, and data types.
order_df.info()

,Order Id,order date (DateOrders),shipping date (DateOrders),Late_delivery_risk,Shipping Mode,Type,Customer Segment,Market,Order Region,Days for shipment (scheduled),Days for shipping (real),Order Item Count,Product Count,Order Quantity,Order Value
0,1,2015-01-01 00:00:00,2015-01-03 00:00:00,0,Standard Class,CASH,Consumer,LATAM,Central America,4,2,1,1,1,239.979996
1,2,2015-01-01 00:21:00,2015-01-04 00:21:00,0,Standard Class,PAYMENT,Consumer,LATAM,South America,4,3,3,3,7,529.380005
2,4,2015-01-01 01:03:00,2015-01-06 01:03:00,1,Standard Class,CASH,Home Office,LATAM,South America,4,5,4,4,14,620.870014


<class 'pandas.DataFrame'>
RangeIndex: 62897 entries, 0 to 62896
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Order Id                       62897 non-null  int64         
 1   order date (DateOrders)        62897 non-null  datetime64[us]
 2   shipping date (DateOrders)     62897 non-null  datetime64[us]
 3   Late_delivery_risk             62897 non-null  int64         
 4   Shipping Mode                  62897 non-null  str           
 5   Type                           62897 non-null  str           
 6   Customer Segment               62897 non-null  str           
 7   Market                         62897 non-null  str           
 8   Order Region                   62897 non-null  str           
 9   Days for shipment (scheduled)  62897 non-null  int64         
 10  Days for shipping (real)       62897 non-null  int64         
 11  Order Item Count          

**Result**

- `order_df` contains one row per eligible order.
- Cancelled shipments were excluded, while the original rows remain in `working_df`.
- Order identifiers are unique and `Late_delivery_risk` is complete.
- Item quantity and order value reconcile with the eligible source rows.
- `order_df` is ready for final validation in Section 5.

### 4.3 Create the complete-week dataset

**Purpose**

* Use `order date (DateOrders)` to assign eligible orders to Monday-to-Sunday calendar weeks.
* Exclude the partial weeks at the beginning and end of the source coverage.
* Create `weekly_df` with one row for every complete calendar week.
* Retain weeks with no eligible orders by recording zero counts.
* Keep only the base weekly counts required for later analysis; rates and forecasts will be calculated in the Analyze phase.

**Applied transformation**


In [27]:
# Use the cleaned source coverage to identify complete weeks.
order_date_column = "order date (DateOrders)"

source_start_date = (
    working_df[order_date_column]
    .min()
    .normalize()
)

source_end_date = (
    working_df[order_date_column]
    .max()
    .normalize()
)


# Identify the first and last complete Monday-to-Sunday weeks.
first_complete_week_start = (
    source_start_date
    + pd.Timedelta(
        days=(-source_start_date.weekday()) % 7
    )
)

last_complete_week_end = (
    source_end_date
    - pd.Timedelta(
        days=(source_end_date.weekday() + 1) % 7
    )
)

last_complete_week_start = (
    last_complete_week_end
    - pd.Timedelta(days=6)
)


# Assign each eligible order to its Monday week start.
order_dates = (
    order_df[order_date_column]
    .dt.normalize()
)

order_week_start = (
    order_dates
    - pd.to_timedelta(
        order_dates.dt.dayofweek,
        unit="D"
    )
)


# Select orders within the complete-week range.
complete_week_mask = order_dates.between(
    first_complete_week_start,
    last_complete_week_end
)

complete_week_starts = pd.date_range(
    start=first_complete_week_start,
    end=last_complete_week_start,
    freq="W-MON",
    name="Week Start"
)


# Aggregate eligible orders by complete week.
weekly_df = (
    order_df.loc[
        complete_week_mask,
        ["Order Id", "Late_delivery_risk"]
    ]
    .assign(
        **{
            "Week Start":
                order_week_start.loc[
                    complete_week_mask
                ]
        }
    )
    .groupby("Week Start")
    .agg(
        **{
            "Eligible Order Count": (
                "Order Id",
                "count"
            ),
            "Late Order Count": (
                "Late_delivery_risk",
                "sum"
            )
        }
    )
    .reindex(
        complete_week_starts,
        fill_value=0
    )
    .rename_axis("Week Start")
    .reset_index()
)


# Store the weekly counts as integers.
weekly_count_columns = [
    "Eligible Order Count",
    "Late Order Count"
]

weekly_df[weekly_count_columns] = (
    weekly_df[weekly_count_columns]
    .astype("int64")
)


In [28]:
# Preview the complete-week dataset.
display(weekly_df.head(3))

# Inspect its dimensions, fields, and data types.
weekly_df.info()

,Week Start,Eligible Order Count,Late Order Count
0,2015-01-05,391,216
1,2015-01-12,380,216
2,2015-01-19,378,213


<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Week Start            160 non-null    datetime64[us]
 1   Eligible Order Count  160 non-null    int64         
 2   Late Order Count      160 non-null    int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 3.9 KB


**Result**

* `weekly_df` contains one row per complete Monday-to-Sunday week.
* The weekly series covers 160 complete weeks, from the week starting 5 January 2015 through the week ending 28 January 2018.
* The partial first and last boundary weeks were excluded.
* The weekly calendar contains no gaps; any week with no eligible orders is retained with zero counts.
* Eligible-order and late-order totals reconcile with `order_df` over the same complete-week period.
* `weekly_df` contains only the three base fields required for later analysis:

  * `Week Start`
  * `Eligible Order Count`
  * `Late Order Count`
* `order_df` and `weekly_df` are ready for final verification in Section 5.


## 5. Verify and save the analysis-ready datasets  
  
This section  
- Perform final verification of both `order_df` and `weekly_df`.  
- Confirm that both datasets contain the structure and fields required for the planned analysis.  
- Save the verified datasets as separate processed files without modifying the original raw data.  
- Document the Process outputs and prepare them for use in the **03 Analyze notebook**.  
  
The verification focuses on the dataset structure and the key fields required for the planned analysis.

### 5.1 Verify the final analysis-ready datasets

The final `order_df` and `weekly_df` are verified before saving:

- Record the dimensions of both datasets.
- Confirm one row per unique `Order Id`.
- Confirm one row per complete Monday-to-Sunday week, with no gaps.
- Confirm that all fields required for the planned analysis are present.
- Check for unresolved missing values in the required fields.
- Reconcile the weekly eligible-order and late-order totals with `order_df`.

**Verification code**

In [29]:
# Define the fields required for the Analyze phase.
order_required_columns = [
    "Order Id",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Late_delivery_risk",
    "Shipping Mode",
    "Type",
    "Customer Segment",
    "Market",
    "Order Region",
    "Days for shipment (scheduled)",
    "Days for shipping (real)",
    "Order Item Count",
    "Product Count",
    "Order Quantity",
    "Order Value"
]

weekly_required_columns = [
    "Week Start",
    "Eligible Order Count",
    "Late Order Count"
]


# Check that all required fields are present.
missing_order_columns = [
    column for column in order_required_columns
    if column not in order_df.columns
]

missing_weekly_columns = [
    column for column in weekly_required_columns
    if column not in weekly_df.columns
]


# Count missing values in the required fields.
order_missing_value_count = (
    int(
        order_df[
            order_required_columns
        ].isna().sum().sum()
    )
    if not missing_order_columns
    else None
)

weekly_missing_value_count = (
    int(
        weekly_df[
            weekly_required_columns
        ].isna().sum().sum()
    )
    if not missing_weekly_columns
    else None
)


In [30]:
# Verify the order-level structure.
order_structure_valid = (
    order_df["Order Id"].notna().all()
    and order_df["Order Id"].is_unique
)


# Verify the complete-week structure.
weekly_calendar_valid = (
    weekly_df["Week Start"].notna().all()
    and weekly_df["Week Start"].is_unique
    and weekly_df["Week Start"].dt.dayofweek.eq(0).all()
    and weekly_df["Week Start"].tolist()
    == complete_week_starts.tolist()
)


# Reconcile weekly totals with the same orders in order_df.
weekly_eligible_total_matches = (
    weekly_df["Eligible Order Count"].sum()
    == int(complete_week_mask.sum())
)

weekly_late_total_matches = (
    weekly_df["Late Order Count"].sum()
    == int(
        order_df.loc[
            complete_week_mask,
            "Late_delivery_risk"
        ].sum()
    )
)


# Confirm that both datasets are ready for analysis.
order_ready = (
    order_structure_valid
    and not missing_order_columns
    and order_missing_value_count == 0
)

weekly_ready = (
    weekly_calendar_valid
    and not missing_weekly_columns
    and weekly_missing_value_count == 0
    and weekly_eligible_total_matches
    and weekly_late_total_matches
)

analysis_ready = order_ready and weekly_ready


In [31]:
# Summarise the final verification results.
verification_summary = pd.DataFrame({
    "Check": [
        "order_df dimensions",
        "One row per unique Order Id",
        "Missing order-level columns",
        "Missing values in order-level fields",
        "weekly_df dimensions",
        "Complete Monday-start weeks with no gaps",
        "Missing weekly columns",
        "Missing values in weekly fields",
        "Eligible order total reconciles",
        "Late order total reconciles",
        "Both datasets ready for the Analyze phase"
    ],
    "Result": [
        (
            f"{order_df.shape[0]:,} rows × "
            f"{order_df.shape[1]} columns"
        ),
        order_structure_valid,
        (
            ", ".join(missing_order_columns)
            if missing_order_columns
            else "None"
        ),
        (
            order_missing_value_count
            if order_missing_value_count is not None
            else "Not checked"
        ),
        (
            f"{weekly_df.shape[0]:,} rows × "
            f"{weekly_df.shape[1]} columns"
        ),
        weekly_calendar_valid,
        (
            ", ".join(missing_weekly_columns)
            if missing_weekly_columns
            else "None"
        ),
        (
            weekly_missing_value_count
            if weekly_missing_value_count is not None
            else "Not checked"
        ),
        weekly_eligible_total_matches,
        weekly_late_total_matches,
        analysis_ready
    ]
})

display(verification_summary)

,Check,Result
0,order_df dimensions,"62,897 rows × 15 columns"
1,One row per unique Order Id,True
2,Missing order-level columns,None
3,Missing values in order-level fields,0
4,weekly_df dimensions,160 rows × 3 columns
5,Complete Monday-start weeks with no gaps,True
6,Missing weekly columns,None
7,Missing values in weekly fields,0
8,Eligible order total reconciles,True
9,Late order total reconciles,True


**Verification result**

- `order_df` contains one row per unique eligible order.
- `weekly_df` contains one row per complete Monday-to-Sunday week, with no calendar gaps.
- All fields required for the planned analysis are present.
- No unresolved missing values remain in the required fields.
- Weekly eligible-order and late-order totals reconcile with `order_df` over the same complete-week period.
- Both datasets are ready to be saved and used in the Analyze phase.

### 5.2 Save and document the Process outputs

The verified datasets are saved as the final Process outputs:

- Save `order_df` as a separate order-level CSV file.
- Save `weekly_df` as a separate complete-week CSV file.
- Keep the original raw dataset unchanged.
- Keep `working_df` as an internal intermediate DataFrame rather than a separate output.
- Record the output locations and dimensions.
- Use both saved files as inputs for the Analyze phase.
- Retain the assessment, treatment, and transformation documentation in Sections 2–4.

In [32]:
# Define the processed output folder and file paths.
output_folder = Path("../data/processed")

order_output_path = (
    output_folder
    / "order_level_analysis_ready.csv"
)

weekly_output_path = (
    output_folder
    / "weekly_analysis_ready.csv"
)

In [ ]:
# Save both datasets only when final verification is successful.
if analysis_ready:
    output_folder.mkdir(parents=True, exist_ok=True)

    order_df.to_csv(
        order_output_path,
        index=False
    )

    weekly_df.to_csv(
        weekly_output_path,
        index=False,
        date_format="%Y-%m-%d"
    )

    # Summarise the saved Process outputs.
    output_summary = pd.DataFrame({
        "Dataset": [
            "Order-level",
            "Complete-week"
        ],
        "File": [
            order_output_path.name,
            weekly_output_path.name
        ],
        "Rows": [
            order_df.shape[0],
            weekly_df.shape[0]
        ],
        "Columns": [
            order_df.shape[1],
            weekly_df.shape[1]
        ]
    })

    print("Process outputs saved successfully.")
    print("Output folder: data/processed")

    display(output_summary)

else:
    print(
        "Datasets were not saved. "
        "Review the verification results in Section 5.1."
    )

**Process outputs**

- `order_level_analysis_ready.csv` contains one row per unique eligible order.
- `weekly_analysis_ready.csv` contains one row per complete Monday-to-Sunday week.
- The original raw dataset was not modified.
- The intermediate `working_df` was not saved as a separate output.
- Data assessment, treatment, and transformation are documented in Sections 2–4.
- Both processed datasets are ready to be loaded in the `03 Analyze` notebook.